# Qwen3 ES Review Fine-Tuning

この notebook は `career_compass/ml/es_review_qwen` を Colab GPU で実行する前提です。

- dataset repo: `saoki0913/career-compass-qwen3-es-review-data`
- adapter repo: `saoki0913/career-compass-qwen3-es-review-lora`
- base model: `Qwen/Qwen3-14B`

Colab の `Secrets` に `HF_TOKEN` を設定してから実行してください。

In [ ]:
DATASET_REPO_ID = "saoki0913/career-compass-qwen3-es-review-data"
ADAPTER_REPO_ID = "saoki0913/career-compass-qwen3-es-review-lora"
BASE_MODEL = "Qwen/Qwen3-14B"


In [ ]:
import os

from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Colab Secrets に HF_TOKEN を設定してください")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGINGFACE_TOKEN"] = hf_token
print("HF_TOKEN loaded")


In [ ]:
%cd /content
!rm -rf /content/career_compass
!git clone https://github.com/saoki0913/career_compass.git
%cd /content/career_compass
!python -m pip install -U pip
!pip install -r ml/es_review_qwen/requirements-train.txt huggingface_hub


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=DATASET_REPO_ID,
    repo_type="dataset",
    local_dir="/content/career_compass/ml/es_review_qwen/data/generated",
    local_dir_use_symlinks=False,
    token=os.environ["HF_TOKEN"],
)
print("dataset downloaded")


In [ ]:
!nvidia-smi

import torch

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print({"gpu": gpu_name})
if "T4" in gpu_name:
    print("Warning: T4 では Qwen3-14B QLoRA が厳しい可能性があります。失敗する場合は 8B へ切り替えます。")


In [ ]:
!python ml/es_review_qwen/scripts/train_unsloth_sft.py --config ml/es_review_qwen/configs/qwen3_14b_lora.json


In [ ]:
!python ml/es_review_qwen/scripts/push_artifact_to_hub.py \
  --source-dir ml/es_review_qwen/outputs/qwen3-14b-es-review-lora \
  --repo-id saoki0913/career-compass-qwen3-es-review-lora \
  --repo-type model \
  --private


In [ ]:
!ls -lah ml/es_review_qwen/outputs/qwen3-14b-es-review-lora
!tar -czf /content/qwen3-14b-es-review-lora.tar.gz -C ml/es_review_qwen/outputs qwen3-14b-es-review-lora
print({"adapter_repo": ADAPTER_REPO_ID, "backup_tar": "/content/qwen3-14b-es-review-lora.tar.gz"})
